# Lab 8-4: Predict Bike Sharing Using MLlib

**Topics**: *Loading CSV into DataFrame, Exploring and Understanding Data, Dropping Columns, Casting Column Types, Splitting Data, Visualizing Data (histograms, group-by, line charts), Feature Engineering (VectorAssembler, VectorIndexer), Training ML Models (GBTRegressor), Evaluating Models (RegressionEvaluator, RMSE), Model Selection (CrossValidator, ParamGridBuilder), ML Pipelines, Visualizing Predictions (scatter plots), Testing Transformers/Estimators.*

This Python notebook demonstrates how to build an end-to-end ML Pipeline using Spark MLlib to preprocess the Bike Sharing dataset, explore and visualize the data, train a machine learning model, and make predictions. The workflow includes data loading, basic data exploration, preprocessing, feature selection, splitting into training and test sets, visualization, model training, evaluation, and hyperparameter tuning.

**Data**: The dataset contains hourly bike rental information from 2011 and 2012 in the Capital Bikeshare system, including weather and calendar features. The data is from Fanaee-T and Gama (2013), hosted by the [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml/datasets/Bike+Sharing+Dataset).

**Goal**: Predict hourly bike rental counts (`cnt`) using features such as day of the week, weather, season, and more. Accurate demand prediction helps optimize resource allocation and improve service.

**Approach**: We use Spark ML Pipelines to streamline the workflow, including feature processing, model training, and evaluation. Model selection and hyperparameter tuning are demonstrated using Cross Validation to improve prediction performance.

## Step 1: Load and split the data

### 1. Set up the volume
- set up a  volume `lab8_4` under schema `spark` of catalog `msbabigdata`

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.lab8_4;

### 2. Download the data and upload to Unity Catalog

- Download the data from [https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip](https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip) 
- unzip it and upload `hour.csv` to the lab volume

### 3. Load the csv file to Spark DataFrame `df`

- then display 5 rows for inspection

In [ ]:
path = "/Volumes/msbabigdata/spark/lab8_4"
df = spark.read.csv(path+"/hour.csv", inferSchema=True, header=True)
df.limit(5).display()

instant | dteday | season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | casual | registered | cnt
1 | 2011-01-01 | 1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 3 | 13 | 16
2 | 2011-01-01 | 1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 8 | 32 | 40
3 | 2011-01-01 | 1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 5 | 27 | 32
4 | 2011-01-01 | 1 | 0 | 1 | 3 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 3 | 10 | 13
5 | 2011-01-01 | 1 | 0 | 1 | 4 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 0 | 1 | 1

**Data description**

From the [UCI ML Repository description](http://archive.ics.uci.edu/ml/datasets/Bike+Sharing+Dataset), we know that the columns have the following meanings.

|field|description|
|--|--|
|instant| record index|
|dteday| date|
|season| season (1:spring, 2:summer, 3:fall, 4:winter)|
|yr| year (0:2011, 1:2012)|
|mnth| month (1 to 12)|
|hr| hour (0 to 23)|
|holiday| whether day is holiday or not|
|weekday| day of the week|
|workingday| if day is neither weekend nor holiday is 1, otherwise is 0.|
|weathersit| 1: Clear, Few clouds, Partly cloudy, Partly cloudy; 2:Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist; 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds; 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog|
|temp| Normalized temperature in Celsius. The values are derived via `(t-t_min)/(t_max-t_min)`, `t_min=-8`, `t_max=+39` (only in hourly scale)|
|atemp| Normalized feeling temperature in Celsius. The values are derived via `(t-t_min)/(t_max-t_min)`, `t_min=-16`, `t_max=+50` (only in hourly scale) |
|hum| Normalized humidity. The values are divided to 100 (max)|
|windspeed| Normalized wind speed. The values are divided to 67 (max)|
|casual | count of casual users|
|registered | count of registered users|
|cnt | count of total rental bikes including both casual and registered|

### 4. Examine Schema and num of rows

- display the schema
- display the number of rows

In [ ]:
df.printSchema()

root
 |-- instant: integer (nullable = true)
 |-- dteday: date (nullable = true)
 |-- season: integer (nullable = true)
 |-- yr: integer (nullable = true)
 |-- mnth: integer (nullable = true)
 |-- hr: integer (nullable = true)
 |-- holiday: integer (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- workingday: integer (nullable = true)
 |-- weathersit: integer (nullable = true)
 |-- temp: double (nullable = true)
 |-- atemp: double (nullable = true)
 |-- hum: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- casual: integer (nullable = true)
 |-- registered: integer (nullable = true)
 |-- cnt: integer (nullable = true)



In [ ]:
df.count()

### 5. Split the dataset for training and validation
- split the data randomly to use 70% for training and 30% for testing (use `.randomSplit([ratio1,ratio2],seed=1234)`) 
- call the resulting dataframes `train` and `test`
- Then verify the count of each dataset

In [ ]:
train, test = df.randomSplit([0.7, 0.3], seed=1234)


In [ ]:
train.count()

In [ ]:
%sql WITH q AS (SELECT hr, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY hr
ORDER BY hr) SELECT `hr`,SUM(`total_cnt`) `column_1c99fe52188` FROM q GROUP BY `hr`

In [ ]:
test.count()

In [ ]:
%sql WITH q AS (SELECT season, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY season
ORDER BY season) SELECT `season`,SUM(`total_cnt`) `column_1c99fe52197` FROM q GROUP BY `season`

In [ ]:
%sql WITH q AS (SELECT mnth, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY mnth
ORDER BY mnth) SELECT `mnth`,SUM(`total_cnt`) `column_1c99fe52200` FROM q GROUP BY `mnth`

### 6. Save `train` and `test` as managed tables

Because we don't want to repeatedly split the dataset and we cannot cache the DataFrames in Serverless Compute, we will persist `train` and `test` as managed tables instead.

- Create a `bikeshare` database.
- Save the two DataFrames as tables `train` and `test`.
- Reload the `train` and `test` DataFrames from the corresponding tables.

In [ ]:
%sql 
CREATE DATABASE IF NOT EXISTS bikeshare;

In [ ]:
train.write.mode("overwrite").saveAsTable("bikeshare.train")
test.write.mode("overwrite").saveAsTable("bikeshare.test")

In [ ]:
#redefined test and train df from table
train = spark.table("bikeshare.train")
test = spark.table("bikeshare.test")

## Step 2. Explore the Relationship Between Features and the Label


Before we begin data engineering, it is useful to examine the relationship between features and the label -- e.g. to see if there is correlation and whether there is a linear relationship. 

- for categorical/binary features, we can plot the group totals
- for continuous features, we may use x-y scatter plots


### 7. Calculate hourly total rental counts and plot it

- first use Spark SQL to calculate hourly averages
- then use Databricks' build-in visualization to create a line chart

> Is the rental account correlated with hour? Is it a linear relationship?

In [ ]:
%sql 
SELECT hr, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY hr
ORDER BY hr;


hr | total_cnt
0 | 24781
1 | 16906
2 | 11607
3 | 5908
4 | 3104
5 | 9910
6 | 40623
7 | 103412
8 | 186461
9 | 112344
10 | 88757
11 | 109496
12 | 133023
13 | 127146
14 | 125947
15 | 126974
16 | 171855
17 | 227773
18 | 221578
19 | 163526
... (24 rows total)

### 8. Do the same for `season` and `mnth`

In [ ]:
%sql 
SELECT season, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY season
ORDER BY season;

season | total_cnt
1 | 339042
2 | 651225
3 | 746708
4 | 595118

In [ ]:
%sql 
SELECT mnth, sum(cnt) AS total_cnt
FROM bikeshare.train
GROUP BY mnth
ORDER BY mnth;

mnth | total_cnt
1 | 99489
2 | 110365
3 | 164860
4 | 187814
5 | 231932
6 | 250323
7 | 247541
8 | 244823
9 | 239578
10 | 226836
11 | 175067
12 | 153465

## Step 3: Build the Feature Pipeline


As a first step, we need to decide what to keep. 

**label**: We want to predict the count of bike rentals, hence the `cnt` column is our label.

**Features**: We can use the rest as features, except these:
* `casual`, `registered`: The `cnt` column equals the sum of the `casual` + `registered` columns. Unless we are interested in separating `casual` and `registered`, these two are not useful to us. 
* `dteday`:  We will discard it because it is well-represented by the other date-related columns `season`, `yr`, `mnth`, and `weekday`. 
* `instant`: This will not be used for analysis.

So, we will only keep the following for analysis:
```
season, yr, mnth, hr, holiday, weekday, workingday, weathersit, temp, atemp, hum, windspeed, cnt 
```

### 9. Drop `instant`,`dteday`,`casual`,`registered` columns

- do it in the way that it can be part of the pipeline (hint: use `SQLTransformer`)
- test it by showing 10 rows after the transformation

In [ ]:
from pyspark.ml.feature import SQLTransformer
st = SQLTransformer(
    statement="""
    SELECT season, yr, mnth, hr, holiday, weekday, workingday,
    weathersit, temp, atemp, hum, windspeed, cnt 
    FROM __THIS__
    """)

#testing - need since we are interactively developing the pipeline
train_st = st.transform(train)
train_st.limit(5).display()

season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | cnt
1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 16
1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 40
1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 32
1 | 0 | 1 | 3 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 13
1 | 0 | 1 | 4 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 1

### 10. Assembled feature columns into a feature vector

- Create a vector assembler `VectorAssembler` that assembles the feature columns into a `rawFeatures` vector. 
- Include all features, but leave out the label column.
- verify it

In [ ]:
from pyspark.ml.feature import VectorAssembler
va = VectorAssembler(
  inputCols=["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"], 
  outputCol="rawFeatures")

#testing
train_va = va.transform(train_st)
train_va.limit(5).display()

season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | cnt | rawFeatures
1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 16 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","0.0","0.0","6.0","0.0","1.0","0.24","0.2879","0.81","0.0"]}
1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 40 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","1.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]}
1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 32 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","2.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]}
1 | 0 | 1 | 3 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 13 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","3.0","0.0","6.0","0.0","1.0","0.24","0.2879","0.75","0.0"]}
1 | 0 | 1 | 4 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.75 | 0.0 | 1 | {"type":"1","size":null,"indices":null,"

In [ ]:
%python
__backend_agg_display_orig = display
__backend_agg_dfs = []
def __backend_agg_display_new(df):
    __backend_agg_df_modules = ["pandas.core.frame", "databricks.koalas.frame", "pyspark.sql.dataframe", "pyspark.pandas.frame", "pyspark.sql.connect.dataframe"]
    if (type(df).__module__ in __backend_agg_df_modules and type(df).__name__ == 'DataFrame') or isinstance(df, list):
        __backend_agg_dfs.append(df)

display = __backend_agg_display_new

def __backend_agg_user_code_fn():
    import base64
    exec(base64.standard_b64decode("dGVzdF9jdi5zZWxlY3QoImNudCIsICJwcmVkaWN0aW9uIikuc2FtcGxlKDAuMSwgc2VlZCA9IDEyMzQpLmRpc3BsYXkoKQ==").decode())

try:
    # run user code
    __backend_agg_user_code_fn()

    #reset display function
    display = __backend_agg_display_orig

    if len(__backend_agg_dfs) > 0:
        # create a temp view
        if type(__backend_agg_dfs[0]).__module__ == "databricks.koalas.frame":
            # koalas dataframe
            __backend_agg_dfs[0].to_spark().createOrReplaceTempView("DatabricksView7c45e06")
        elif type(__backend_agg_dfs[0]).__module__ == "pandas.core.frame" or isinstance(__backend_agg_dfs[0], list):
            # pandas dataframe
            spark.createDataFrame(__backend_agg_dfs[0]).createOrReplaceTempView("DatabricksView7c45e06")
        else:
            __backend_agg_dfs[0].createOrReplaceTempView("DatabricksView7c45e06")
        #run backend agg
        display(spark.sql("""WITH q AS (select * from DatabricksView7c45e06) SELECT `cnt`,`prediction` FROM q"""))
    else:
        displayHTML("dataframe no longer exists. If you're using dataframe.display(), use display(dataframe) instead.")


finally:
    spark.sql("drop view if exists DatabricksView7c45e06")
    display = __backend_agg_display_orig
    del __backend_agg_display_new
    del __backend_agg_display_orig
    del __backend_agg_dfs
    del __backend_agg_user_code_fn



### 11. Label the categorical features with `VectorIndexer`

So far the variables are all presumed ordinal variables but tree-based models can handle categorical featurs such as Hour or Month. 

- Use `VectorIndexor` to designate all features with 24 or fewer distinct values as categorical. 
- the indexed vector will be called `features`
- Verify it by displaying 5 rows.
- and display the metadata for `features` using  `DataFrame.schema.fields[n].metadata`



In [ ]:
from pyspark.ml.feature import VectorIndexer

vi = VectorIndexer(
  inputCol="rawFeatures", 
  outputCol="features",
  maxCategories=24) #24hrs

#testing
train_vi = vi.fit(train_va).transform(train_va)
train_vi.limit(5).display()

season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | cnt | rawFeatures | features
1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 16 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","0.0","0.0","6.0","0.0","1.0","0.24","0.2879","0.81","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","0.0","0.0","6.0","0.0","0.0","0.24","0.2879","0.81","0.0"]}
1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 40 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","1.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","1.0","0.0","6.0","0.0","0.0","0.22","0.2727","0.8","0.0"]}
1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 32 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","2.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]} | {"type":"1","size":null,"in

In [ ]:
train_vi.schema.fields[-1].metadata

In [ ]:
train_vi.limit(5).toPandas()

## Step 3. Build and Train a ML Pipeline

Now that we have understood our data and prepared it as a DataFrame with numeric values, let's learn an ML model to predict bike sharing rentals in the future.  Most ML algorithms expect to predict a single "label" column (`cnt` for our dataset) using a single "features" column of feature vectors.  


### 12. Define the GBTRegressor. 

- `GBTRegressor` takes `features` as feature vectors and `cnt` as labels. 
- Save the resulting GBTRegressor as `gbt`.
- test it

In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
  labelCol="cnt",
  featuresCol="features")

#testing 
gbt_model = gbt.fit(train_vi)
gbt_model.transform(train_vi).limit(5).display()

season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | cnt | rawFeatures | features | prediction
1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 16 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","0.0","0.0","6.0","0.0","1.0","0.24","0.2879","0.81","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","0.0","0.0","6.0","0.0","0.0","0.24","0.2879","0.81","0.0"]} | 47.066318911586905
1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 40 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","1.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","1.0","0.0","6.0","0.0","0.0","0.22","0.2727","0.8","0.0"]} | 38.89464571176288
1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 32 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","2.0","0.0","6.0","0.0","1.0","0.22


### 13. Put together the ML pipeline

We will put together a simple Pipeline from feature engineering to model fitting:
* SQLTransformer: Dropping columns
* VectorAssembler: Assemble the feature columns into a feature vector.
* VectorIndexer: Label categorical columns
* GBTRegressor: This will use the Gradient-Boosted Trees (GBT) algorithm to learn how to predict rental counts from the feature vectors.
|

In [ ]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[st, va, vi, gbt])

### 14. Train the pipeline and use the trained pipeline to make predictions

- Train it on the training dataset
- Apply the trained pipeline on the `train` and `test` datasets
- Display prediction results for inspection

In [ ]:
pl_model = pipeline.fit(train)
train_pl = pl_model.transform(train)
test_pl = pl_model.transform(test)

train_pl.limit(5).display()

season | yr | mnth | hr | holiday | weekday | workingday | weathersit | temp | atemp | hum | windspeed | cnt | rawFeatures | features | prediction
1 | 0 | 1 | 0 | 0 | 6 | 0 | 1 | 0.24 | 0.2879 | 0.81 | 0.0 | 16 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","0.0","0.0","6.0","0.0","1.0","0.24","0.2879","0.81","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","0.0","0.0","6.0","0.0","0.0","0.24","0.2879","0.81","0.0"]} | 47.066318911586905
1 | 0 | 1 | 1 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 40 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","1.0","0.0","6.0","0.0","1.0","0.22","0.2727","0.8","0.0"]} | {"type":"1","size":null,"indices":null,"values":["0.0","0.0","0.0","1.0","0.0","6.0","0.0","0.0","0.22","0.2727","0.8","0.0"]} | 38.89464571176288
1 | 0 | 1 | 2 | 0 | 6 | 0 | 1 | 0.22 | 0.2727 | 0.8 | 0.0 | 32 | {"type":"1","size":null,"indices":null,"values":["1.0","0.0","1.0","2.0","0.0","6.0","0.0","1.0","0.22

In [ ]:
test_pl.select("cnt","prediction").limit(10).display()

cnt | prediction
14 | 47.980165798186775
56 | 170.65841815100617
93 | 169.45086121248733
67 | 180.7323038891936
6 | 28.809885375392707
2 | -2.265654938983962
8 | 101.66498444347137
75 | 163.17544442063996
53 | 118.46228164088241
1 | -3.5535658296879737

### 15. Create an evaluator `e` and use it for evaluations.

- Use RMSE as the evaluation metric
- Use it to evaluate predictions on the training and test datasets

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

e = RegressionEvaluator(
  labelCol="cnt",
  predictionCol="prediction",
  metricName="rmse")

e.evaluate(train_pl)

In [ ]:
e.evaluate(test_pl)

## Step 4. Tune the Model using Hypter Parameter Tuning and CrossValidator




### 16. Create a parameter grid `paramGrid`

We will explore one parameter for GTB

  - `maxDepth`: max depth of each decision tree in the GBT ensemble; 
  - we will explore two levels: 5 (default) and 8
  - Another parameter you may probe is maxIter: e.g., 10 (default) and 20.
 


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

param_grid = ParamGridBuilder() \
  .addGrid(gbt.maxDepth, [2, 5]) \
  .addGrid(gbt.maxIter, [10, 20]) \
  .build()

### 17. Create a CrossValidator `cv` and Train/Apply it

- Use the pipeline as as estimator to be cross-validated
- Use `e` as evaluator 
- Use `paramGrid` as parameter maps.

Then train the cross-validator, and use it on the test data.

Before you train the cross validator, you need to set up an environment variable, telling Databricks where to save temporary files.

In [ ]:
import os

# Define your UC Volume path
# Format: /Volumes/<catalog>/<schema>/<volume>/<optional_path>


# Set the environment variable for SparkML caching
os.environ['SPARKML_TEMP_DFS_PATH'] = path

In [ ]:
cv = CrossValidator(
  estimator=pipeline,
  estimatorParamMaps=param_grid,
  evaluator=e,
  numFolds=3)
cv_model = cv.fit(train)
train_cv = cv_model.transform(train)
e.evaluate(train_cv)

In [ ]:
test_cv = cv_model.transform(test)
e.evaluate(test_cv)

### 18. View the best model parameters

- You can access the crossValidator's best model using `.bestModel`
- You can access a pipeline's stage X parameter Y using `.stages[x].explainParam[paramName]` (a single paramter) or `.stages[x].explainParams` (all parameters)



In [ ]:
cv_model.bestModel.stages[3].explainParam("maxDepth")

In [ ]:
cv_model.bestModel.stages[3].explainParam("maxIter")

### 19. Visualize final predictions using a Scatter Plot

- use 10% sample of the predictions on the test data (use `.sample(False,0.1, seed=1234)`)
- focus on the  ()`cnt`,`prediction`) pair in the scatter plot

In [ ]:
test_cv.select("cnt", "prediction").sample(0.1, seed = 1234).display()

cnt | prediction
14 | 47.980165798186775
93 | 169.45086121248733
61 | 70.83322002005022
19 | 21.081420714361055
95 | 130.1243374088209
59 | 72.57725192055494
48 | 76.34607380011843
5 | 14.730019928415288
188 | 163.15017607085142
3 | 11.011862488984649
6 | 20.54479490904881
139 | 151.04854337838296
118 | 134.6298206219477
105 | 95.9990718530879
33 | 80.35285935496708
13 | -30.124458449080258
112 | 136.15328099888913
1 | -3.5535658296879737
63 | 64.64538759339555
7 | 22.023286247384323
... (510 rows total)

### 20. Clean up

- drop the database and volumes created by this lab


In [ ]:
%sql 
DROP VOLUME IF EXISTS msbabigdata.spark.lab8_4;
DROP DATABASE IF EXISTS bikeshare CASCADE;

This is the end of the lab